In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
os.environ['AICORE_CLIENT_ID'] = AICORE_CLIENT_ID
os.environ['AICORE_CLIENT_SECRET'] = AICORE_CLIENT_SECRET
os.environ['AICORE_AUTH_URL'] = AICORE_AUTH_URL
os.environ['AICORE_BASE_URL'] = AICORE_BASE_URL
os.environ['AICORE_RESOURCE_GROUP'] = AICORE_RESOURCE_GROUP

https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/get-auth-token

In [5]:
import requests
from datetime import datetime, timedelta

# Check if we already have a valid token
def get_or_refresh_token():
    """Get existing token or request a new one if needed"""
    
    # Check if token already exists in the kernel
    if 'access_token' in globals() and 'token_expiry' in globals():
        # Check if token is still valid (with 1 minute buffer)
        if datetime.now() < token_expiry:
            print("Using cached token (still valid)")
            return access_token, token_expiry
    
    # Get environment variables for authentication
    AUTH_URL = os.getenv("AICORE_AUTH_URL")
    CLIENT_ID = os.getenv("AICORE_CLIENT_ID") 
    CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
    
    print("Requesting new token...")
    
    # Make the POST request to get OAuth token
    response = requests.post(
        f"{AUTH_URL}/oauth/token",
        headers={
            'content-type': 'application/x-www-form-urlencoded'
        },
        data={
            'grant_type': 'client_credentials',
            'client_id': CLIENT_ID,
            'client_secret': CLIENT_SECRET
        }
    )
    
    # Check if request was successful
    if response.status_code == 200:
        token_data = response.json()
        new_token = token_data.get('access_token')
        expires_in = token_data.get('expires_in', 3600)  # Default to 1 hour
        
        # Calculate expiry time with 1-minute buffer
        new_expiry = datetime.now() + timedelta(seconds=expires_in - 60)
        
        print("Token obtained successfully!")
        print(f"Token expires in: {expires_in} seconds ({expires_in/60:.1f} minutes)")
        print(f"Access Token: {new_token[:50]}..." if new_token else "No access token in response")
        
        return new_token, new_expiry
    else:
        print(f"Error: {response.status_code}")
        print(f"Response: {response.text}")
        return None, None

# Get or refresh the token
access_token, token_expiry = get_or_refresh_token()

# Store the token for later use
if access_token:
    os.environ['ACCESS_TOKEN'] = access_token

Requesting new token...
Token obtained successfully!
Token expires in: 43199 seconds (720.0 minutes)
Access Token: eyJ0eXAiOiJKV1QiLCJqaWQiOiJGd09kM1FUUGVxRzZwaHRFSU...


https://help.sap.com/docs/sap-ai-core/sap-ai-core-service-guide/example-payloads-for-inferencing-sap-ai-core-hosted

In [12]:
# Make chat request using the access token
import json

# Get environment variables
BASE_URL = os.getenv("AICORE_BASE_URL")
RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")

# Construct the correct deployment URL
DEPLOYMENT_URL = f"{BASE_URL}/inference/deployments/d1cd5340a145e7a9"

# Optional: Enable thinking mode for more detailed reasoning (increases response time)
USE_THINKING = False  # Set to True to enable thinking mode

# Chat request payload
chat_payload = {
    "model": "cohere--command-a-reasoning",
    "stream": False,
    "frequency_penalty": 0.8,
    "thinking": {
        "type": "disabled" if not USE_THINKING else "enabled"
    },
    "messages": [
        {
            "role": "user",
            "content": "What are you?"
        }
    ]
}

# Make the chat request using the working /chat endpoint
print(f"Making request to: {DEPLOYMENT_URL}/chat")
print(f"Thinking mode: {'enabled' if USE_THINKING else 'disabled'}")

chat_response = requests.post(
    f"{DEPLOYMENT_URL}/chat",
    headers={
        'AI-Resource-Group': RESOURCE_GROUP,
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {access_token}'
    },
    json=chat_payload
)

print(f"Response Status: {chat_response.status_code}")

# Check response
if chat_response.status_code == 200:
    chat_result = chat_response.json()
    print("Chat request successful!")
    print(json.dumps(chat_result, indent=2))
else:
    print(f"Error: {chat_response.status_code}")
    print(f"Response: {chat_response.text}")

# Store the response for later use
if chat_response.status_code == 200:
    chat_data = chat_result

Making request to: https://api.ai.prod.ap-southeast-2.aws.ml.hana.ondemand.com/v2/inference/deployments/d1cd5340a145e7a9/chat
Thinking mode: disabled
Response Status: 200
Chat request successful!
{
  "finish_reason": "COMPLETE",
  "id": "04bfd803-e8bb-4eb2-aebc-3617e45364d2",
  "message": {
    "content": [
      {
        "text": "I'm **Command**, a large language model (LLM) built by **Cohere**. My purpose is to assist with a wide range of tasks, from answering questions and generating text to solving problems and providing creative content. I\u2019m designed to be conversational, informative, and helpful, adapting to your needs and context.\n\nHere\u2019s a quick overview of what I can do:\n- Answer questions on diverse topics (science, history, technology, etc.).\n- Generate text, stories, or ideas.\n- Help with writing, editing, or summarizing content.\n- Provide explanations, tutorials, or step-by-step guidance.\n- Assist with coding, math, or problem-solving.\n\nLet me know how 

In [13]:
# Extract the text response (handles both thinking enabled/disabled)
response_text = None
for content in chat_data["message"]["content"]:
    if content.get("type") == "text":
        response_text = content["text"]
        break

response_text

"I'm **Command**, a large language model (LLM) built by **Cohere**. My purpose is to assist with a wide range of tasks, from answering questions and generating text to solving problems and providing creative content. I’m designed to be conversational, informative, and helpful, adapting to your needs and context.\n\nHere’s a quick overview of what I can do:\n- Answer questions on diverse topics (science, history, technology, etc.).\n- Generate text, stories, or ideas.\n- Help with writing, editing, or summarizing content.\n- Provide explanations, tutorials, or step-by-step guidance.\n- Assist with coding, math, or problem-solving.\n\nLet me know how I can help you today! 😊"